In [ ]:
import requests
import time
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
import os
from dotenv import load_dotenv

load_dotenv()

API_TOKEN = os.getenv("GFW_API_TOKEN")

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

BASE_4WINGS = "https://gateway.api.globalfishingwatch.org/v3/4wings/report"
BASE_EVENTS = "https://gateway.api.globalfishingwatch.org/v3/events"

In [ ]:
def generate_month_ranges(start, end):
    ranges = []
    current = start
    
    while current < end:
        next_month = current + relativedelta(months=1)
        ranges.append((current.strftime("%Y-%m-%d"), next_month.strftime("%Y-%m-%d")))
        current = next_month
    
    return ranges

def download_events_month(start_date, end_date, out_file):

    all_entries = []
    offset = 0
    limit = 100000  # max safe value

    while True:
        print(f"Fetching offset {offset}")

        params = {
            "limit": limit,
            "offset": offset
        }

        payload = {
            "datasets": ["public-global-port-visits-events:latest"],
            "types": ["PORT_VISIT"],
            "vesselTypes": ["CARGO", "CONTAINER"],
            "startDate": start_date,
            "endDate": end_date
        }

        response = requests.post(
            BASE_EVENTS,
            params=params,
            json=payload,
            headers=HEADERS
        )

        # if response.status_code != 200:
        #     print(response.text)
        #     raise Exception("Events API failed")

        data = response.json()

        entries = data.get("entries", [])

        if not entries:
            break

        all_entries.extend(entries)

        print(f"Collected: {len(all_entries) * 100 / data.get('total')}")

        # Stop if last batch
        if len(entries) < limit:
            break

        offset += limit
        time.sleep(1)  # avoid rate limits

    # Convert to DataFrame (flatten important fields)
    processed = []

    for e in all_entries:
        try:
            processed.append({
                "ship_id": e["vessel"]["ssvid"],
                "ship_type": e["vessel"]["type"],
                "port_id": e["port_visit"]["startAnchorage"]["id"],
                "port_name": e["port_visit"]["startAnchorage"]["name"],
                "lat": e["port_visit"]["startAnchorage"]["lat"],
                "lon": e["port_visit"]["startAnchorage"]["lon"],
                "entry_time": e["start"],
                "exit_time": e["end"],
                "duration_hrs": e["port_visit"]["durationHrs"]
            })
        except Exception as err:
            continue  # skip malformed entries

    df = pd.DataFrame(processed)
    df.to_csv(out_file, index=False)

    print(f"Saved {len(df)} rows → {out_file}")

def download_events_year(start_date, end_date, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    
    month_ranges = generate_month_ranges(start_date, end_date)
    
    files = []
    
    for start, end in month_ranges:
        print(f"Events: {start} → {end}")
        
        filename = f"{out_dir}/events_{start}.csv"
        download_events_month(start, end, filename)
        
        files.append(filename)
    
    return files

def merge_csv(files, output_file):
    df_list = []
    
    for f in files:
        try:
            df = pd.read_csv(f)
            df_list.append(df)
        except:
            print(f"Skipping {f}")
    
    combined = pd.concat(df_list, ignore_index=True)
    combined.to_csv(output_file, index=False)
    
    print(f"Saved merged file: {output_file}")

In [9]:
# Period 1
start_1 = datetime(2015, 1, 1)
end_1   = datetime(2016, 1, 1)

# Period 2
start_2 = datetime(2025, 1, 1)
end_2   = datetime(2026, 1, 1)

In [10]:
# Events
files_ev_2015 = download_events_year(start_1, end_1, "data/events_2015")

# Merge
merge_csv(files_ev_2015, "data/events_2015_full.csv")

Events: 2015-01-01 → 2015-02-01
Fetching offset 0
Collected: 54.17499607231279
Fetching offset 100000
Collected: 100.0
Saved 184587 rows → data/events_2015/events_2015-01-01.csv
Events: 2015-02-01 → 2015-03-01
Fetching offset 0
Collected: 62.984984379723876
Fetching offset 100000
Collected: 100.0
Saved 158768 rows → data/events_2015/events_2015-02-01.csv
Events: 2015-03-01 → 2015-04-01
Fetching offset 0
Collected: 54.24729170396168
Fetching offset 100000
Collected: 100.0
Saved 184341 rows → data/events_2015/events_2015-03-01.csv
Events: 2015-04-01 → 2015-05-01
Fetching offset 0
Collected: 54.039740825403
Fetching offset 100000
Collected: 100.0
Saved 185049 rows → data/events_2015/events_2015-04-01.csv
Events: 2015-05-01 → 2015-06-01
Fetching offset 0
Collected: 53.38999791779008
Fetching offset 100000
Collected: 100.0
Saved 187301 rows → data/events_2015/events_2015-05-01.csv
Events: 2015-06-01 → 2015-07-01
Fetching offset 0
Collected: 52.784933068704866
Fetching offset 100000
Collecte

In [7]:
# Events
files_ev_2025 = download_events_year(start_2, end_2, "data/events_2025")

# Merge
merge_csv(files_ev_2025, "data/events_2025_full.csv")

Events: 2025-01-01 → 2025-02-01
Fetching offset 0
Collected: 51.252351201611376
Fetching offset 100000
Collected: 100.0
Saved 195113 rows → data/events_2025/events_2025-01-01.csv
Events: 2025-02-01 → 2025-03-01
Fetching offset 0
Collected: 55.402582868413326
Fetching offset 100000
Collected: 100.0
Saved 180497 rows → data/events_2025/events_2025-02-01.csv
Events: 2025-03-01 → 2025-04-01
Fetching offset 0
Collected: 47.76073780787765
Fetching offset 100000
Collected: 95.5214756157553
Fetching offset 200000
Collected: 100.0
Saved 209377 rows → data/events_2025/events_2025-03-01.csv
Events: 2025-04-01 → 2025-05-01
Fetching offset 0
Collected: 48.658248789626064
Fetching offset 100000
Collected: 97.31649757925213
Fetching offset 200000
Collected: 100.0
Saved 205515 rows → data/events_2025/events_2025-04-01.csv
Events: 2025-05-01 → 2025-06-01
Fetching offset 0
Collected: 47.017890307261915
Fetching offset 100000
Collected: 94.03578061452383
Fetching offset 200000
Collected: 100.0
Saved 2126